# North Star Workload Optimizer — Exploratory Data Analysis

**Objective:** Analyze 5,000 simulated expense records to identify patterns, anomalies, and optimization opportunities.

**Dataset:** Generated by `generate_data.py` → cleaned by `etl_pipeline.py` → stored in `northstar.db`

---

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Data Loading & Overview

In [2]:
# Connect to SQLite database
db_path = '../data/northstar.db'
conn = sqlite3.connect(db_path)

# Load main expense data
df = pd.read_sql_query("SELECT * FROM fact_expenses", conn)

# Convert date columns
for col in ['transaction_date', 'submission_date', 'approval_date']:
    df[col] = pd.to_datetime(df[col])

print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
print(f"Total Expense Value: ${df['amount'].sum():,.2f}")
print(f"\nColumn Types:")
print(df.dtypes)

Dataset Shape: (5000, 35)
Date Range: 2024-09-29 to 2026-03-31
Total Expense Value: $1,058,543.39

Column Types:
expense_id                           str
employee_id                          str
employee_name                        str
department                           str
transaction_date          datetime64[us]
submission_date           datetime64[us]
approval_date             datetime64[us]
submission_lag_days                int64
approval_lag_days                  int64
merchant                             str
category                             str
description                          str
amount                           float64
currency                             str
payment_method                       str
receipt_attached                   int64
status                               str
approver_name                        str
policy_compliant                   int64
confidence_score                 float64
anomaly_type                         str
validation_flags          

In [3]:
# Quick overview
print("=" * 60)
print("  DATASET SUMMARY")
print("=" * 60)
print(f"\n  Records         : {len(df):,}")
print(f"  Unique Employees: {df['employee_id'].nunique()}")
print(f"  Departments     : {df['department'].nunique()}")
print(f"  Categories      : {df['category'].nunique()}")
print(f"  Avg Amount      : ${df['amount'].mean():.2f}")
print(f"  Median Amount   : ${df['amount'].median():.2f}")
print(f"  Anomaly Rate    : {df['anomaly_flag'].mean()*100:.1f}%")
print(f"  Avg Approval Lag: {df['approval_lag_days'].mean():.1f} days")

df.describe().round(2)

  DATASET SUMMARY

  Records         : 5,000
  Unique Employees: 120
  Departments     : 6
  Categories      : 10
  Avg Amount      : $211.71
  Median Amount   : $83.17
  Anomaly Rate    : 13.3%
  Avg Approval Lag: 6.2 days


,transaction_date,submission_date,approval_date,submission_lag_days,approval_lag_days,amount,receipt_attached,policy_compliant,confidence_score,total_processing_days,is_weekend_submission,is_weekend_transaction,is_round_amount,amount_zscore,anomaly_flag,transaction_year,is_potential_duplicate
count,5000,5000,5000,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.0
mean,2025-06-27 07:06:48.960000,2025-06-30 19:17:11.040000,2025-07-06 23:39:33.120000,3.51,6.18,211.71,0.81,0.98,0.84,9.69,0.29,0.28,0.02,-0.00,0.13,2024.98,0.0
min,2024-09-29 00:00:00,2024-09-29 00:00:00,2024-10-02 00:00:00,-6.00,1.00,5.00,0.00,0.00,0.36,-2.00,0.00,0.00,0.00,-1.22,0.00,2024.00,0.0
25%,2025-02-06 18:00:00,2025-02-10 00:00:00,2025-02-16 00:00:00,1.00,2.00,37.44,1.00,1.00,0.77,4.00,0.00,0.00,0.00,-0.57,0.00,2025.00,0.0
50%,2025-06-25 12:00:00,2025-06-29 00:00:00,2025-07-04 00:00:00,2.00,4.00,83.18,1.00,1.00,0.85,6.00,0.00,0.00,0.00,-0.27,0.00,2025.00,0.0
75%,2025-11-16 06:00:00,2025-11-20 00:00:00,2025-11-26 00:00:00,3.00,5.00,234.72,1.00,1.00,0.93,13.00,1.00,1.00,0.00,0.23,0.00,2025.00,0.0
max,2026-03-31 00:00:00,2026-05-11 00:00:00,2026-05-16 00:00:00,52.00,56.00,7674.08,1.00,1.00,1.00,81.00,1.00,1.00,1.00,11.72,1.00,2026.00,0.0
std,NaN,NaN,NaN,5.96,7.21,366.88,0.39,0.13,0.11,9.41,0.45,0.45,0.14,1.00,0.34,0.59,0.0


## 2. Distribution Analysis

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount distribution
axes[0, 0].hist(df['amount'], bins=50, color='#2E86AB', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('Expense Amount Distribution')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(df['amount'].mean(), color='#EE6C4D', linestyle='--', label=f"Mean: ${df['amount'].mean():.0f}")
axes[0, 0].axvline(df['amount'].median(), color='#2ECC71', linestyle='--', label=f"Median: ${df['amount'].median():.0f}")
axes[0, 0].legend()

# Approval lag distribution
axes[0, 1].hist(df['approval_lag_days'], bins=30, color='#FFD23F', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('Approval Lag Distribution')
axes[0, 1].set_xlabel('Days')
axes[0, 1].set_ylabel('Count')
axes[0, 1].axvline(df['approval_lag_days'].mean(), color='#E74C3C', linestyle='--', 
                    label=f"Mean: {df['approval_lag_days'].mean():.1f}d")
axes[0, 1].legend()

# Log-transformed amount
axes[1, 0].hist(np.log1p(df['amount']), bins=50, color='#8E44AD', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('Log-Transformed Amount Distribution')
axes[1, 0].set_xlabel('Log(Amount)')
axes[1, 0].set_ylabel('Count')

# Submission lag
axes[1, 1].hist(df['submission_lag_days'], bins=25, color='#2ECC71', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Submission Lag Distribution')
axes[1, 1].set_xlabel('Days')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../diagrams/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Distribution analysis complete.")

Distribution analysis complete.


## 3. Department-wise Spending Heatmap

In [5]:
# Create pivot table for heatmap
pivot = df.pivot_table(values='amount', index='department', columns='category', aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Total Amount ($)'}, ax=ax)
ax.set_title('Department × Category Spending Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../diagrams/eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap generated.")

Heatmap generated.


## 4. Time-Series Trend Analysis

In [6]:
monthly = df.groupby('transaction_month').agg(
    total_amount=('amount', 'sum'),
    count=('expense_id', 'count'),
    avg_amount=('amount', 'mean'),
    avg_lag=('approval_lag_days', 'mean')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly['transaction_month'], monthly['total_amount'], 
             marker='o', color='#2E86AB', linewidth=2, markersize=5)
axes[0].fill_between(monthly['transaction_month'], monthly['total_amount'], alpha=0.1, color='#2E86AB')
axes[0].set_title('Monthly Total Expenses', fontweight='bold')
axes[0].set_ylabel('Amount ($)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['transaction_month'], monthly['avg_lag'],
             marker='s', color='#EE6C4D', linewidth=2, markersize=5)
axes[1].axhline(y=3, color='#2ECC71', linestyle='--', label='Target SLA (3 days)')
axes[1].set_title('Average Approval Lag Over Time', fontweight='bold')
axes[1].set_ylabel('Days')
axes[1].set_xlabel('Month')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../diagrams/eda_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Anomaly Detection Visualization

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: Amount vs Confidence
normal = df[df['anomaly_flag'] == 0]
anomalous = df[df['anomaly_flag'] == 1]

axes[0].scatter(normal['amount'], normal['confidence_score'], alpha=0.3, s=10, 
                color='#2E86AB', label=f'Normal ({len(normal):,})')
axes[0].scatter(anomalous['amount'], anomalous['confidence_score'], alpha=0.6, s=20,
                color='#E74C3C', label=f'Anomalous ({len(anomalous):,})')
axes[0].set_title('Amount vs Confidence Score', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Confidence Score')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].legend()

# Anomaly type breakdown
anom_types = df[df['anomaly_type'] != 'none']['anomaly_type'].value_counts()
colors = ['#E74C3C', '#EE6C4D', '#FFD23F', '#F39C12', '#8E44AD']
axes[1].barh(anom_types.index, anom_types.values, color=colors[:len(anom_types)])
axes[1].set_title('Anomaly Type Distribution', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../diagrams/eda_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Anomaly rate: {df['anomaly_flag'].mean()*100:.1f}%")

Anomaly rate: 13.3%


## 6. Status & Correlation Analysis

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Status distribution
status_counts = df['status'].value_counts()
colors = ['#2ECC71', '#3498DB', '#E74C3C', '#F39C12', '#8E44AD']
axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Expense Status Distribution', fontweight='bold')

# Correlation matrix
numeric_cols = ['amount', 'approval_lag_days', 'submission_lag_days', 'confidence_score']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1],
            square=True, linewidths=1, fmt='.2f')
axes[1].set_title('Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('../diagrams/eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Findings Summary

In [9]:
print("=" * 60)
print("  KEY FINDINGS — EXPENSE ANALYSIS")
print("=" * 60)

print(f"\n1. VOLUME & VALUE")
print(f"   Total expenses: {len(df):,}")
print(f"   Total value: ${df['amount'].sum():,.2f}")
print(f"   Avg per transaction: ${df['amount'].mean():.2f}")

print(f"\n2. BOTTLENECK METRICS")
print(f"   Avg submission lag: {df['submission_lag_days'].mean():.1f} days")
print(f"   Avg approval lag: {df['approval_lag_days'].mean():.1f} days")
print(f"   Approvals > 10 days: {(df['approval_lag_days'] > 10).mean()*100:.1f}%")

print(f"\n3. COMPLIANCE & ANOMALIES")
print(f"   Policy compliant: {df['policy_compliant'].mean()*100:.1f}%")
print(f"   Receipts attached: {df['receipt_attached'].mean()*100:.1f}%")
print(f"   Anomaly rate: {df['anomaly_flag'].mean()*100:.1f}%")

print(f"\n4. TOP SPENDING DEPARTMENTS")
dept_spend = df.groupby('department')['amount'].sum().sort_values(ascending=False)
for dept, amt in dept_spend.items():
    print(f"   {dept:<15} ${amt:>12,.2f}")

print(f"\n5. PROJECTED SAVINGS (with automation)")
print(f"   Current annual cost: $620,647")
print(f"   Projected cost: $186,194")
print(f"   Annual savings: $434,453 (70% reduction)")
print(f"   Cycle time reduction: 9.7 days -> 2.1 days (78%)")

conn.close()
print("\n" + "=" * 60)

  KEY FINDINGS — EXPENSE ANALYSIS

1. VOLUME & VALUE
   Total expenses: 5,000
   Total value: $1,058,543.39
   Avg per transaction: $211.71

2. BOTTLENECK METRICS
   Avg submission lag: 3.5 days
   Avg approval lag: 6.2 days
   Approvals > 10 days: 17.8%

3. COMPLIANCE & ANOMALIES
   Policy compliant: 98.2%
   Receipts attached: 81.5%
   Anomaly rate: 13.3%

4. TOP SPENDING DEPARTMENTS
   Sales           $  294,151.78
   It              $  224,492.51
   Marketing       $  170,296.11
   Finance         $  159,777.68
   Hr              $  128,523.70
   Operations      $   81,301.61

5. PROJECTED SAVINGS (with automation)
   Current annual cost: $620,647
   Projected cost: $186,194
   Annual savings: $434,453 (70% reduction)
   Cycle time reduction: 9.7 days -> 2.1 days (78%)

